In [ ]:
import onnx
from onnx import helper

m = onnx.load("../training/onnx_out/vits-l1.onnx")

for i, node in enumerate(m.graph.node):
    attrs = {
        a.name: helper.get_attribute_value(a)
        for a in node.attribute
    }

    if node.op_type == "Reshape" or "allowzero" in attrs:
        print(
            i,
            "name =", node.name,
            "op =", node.op_type,
            "allowzero =", attrs.get("allowzero", "<default>"),
            "shape_input =", node.input[1] if len(node.input) > 1 else None,
        )

In [ ]:
for node in m.graph.node:
    if node.op_type != "Reshape":
        continue

    for attr in node.attribute:
        if attr.name == "allowzero" and attr.i == 1:
            print("Fixing:", node.name)
            attr.i = 0

onnx.checker.check_model(m)
onnx.save(m, "../training/onnx_out/vits-l1.onnx")